In [ ]:
# Cell 1: Verify GPU
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=True)
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Clone GitHub repo
import os
import subprocess

REPO_URL  = 'https://github.com/krantiprakash/VisDrone2019-Multi-Object-Detection-and-Tracking.git'
REPO_NAME = 'VisDrone2019-Multi-Object-Detection-and-Tracking'
WORK_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(WORK_DIR):
    result = subprocess.run(
        ['git', 'clone', REPO_URL, WORK_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Git clone failed. Check output above.')
else:
    result = subprocess.run(
        ['git', '-C', WORK_DIR, 'pull'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError('Git pull failed. Check output above.')

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3: Install dependencies
import os

REPO_NAME = 'VisDrone2019-Multi-Object-Detection-and-Tracking'
WORK_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(WORK_DIR):
    raise RuntimeError(f'Repo not found at {WORK_DIR}. Ensure Cell 2 ran successfully.')

os.chdir(WORK_DIR)

!pip install -r requirements.txt -q
!pip install git+https://github.com/KaiyangZhou/deep-person-reid.git -q
!pip install git+https://github.com/JonathonLuiten/TrackEval.git -q
print('All dependencies installed.')

In [ ]:
# Cell 4: Verify configs load correctly
# Note: dataset path verification happens in Cell 5 after copy is done.
import os
import sys
import importlib

REPO_NAME = 'VisDrone2019-Multi-Object-Detection-and-Tracking'
WORK_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(WORK_DIR):
    raise RuntimeError(f'Repo not found at {WORK_DIR}. Ensure Cell 2 ran successfully.')

os.chdir(WORK_DIR)

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import configs.paths
importlib.reload(configs.paths)
from configs.paths import get_paths

paths = get_paths()
print('Resolved paths:')
for key, val in paths.items():
    print(f'  {key:<20} : {val}')

In [ ]:
# Cell 5: Copy dataset to writable location and convert annotations to YOLO format
#
# Kaggle /kaggle/input/ is read-only — labels cannot be written there.
# We copy images and annotations to /kaggle/working/Dataset/ (writable, 20GB available).
# Labels are written alongside copied images so YOLO resolves them correctly
# by replacing 'images' with 'labels' in the image path.

import os
import sys
import importlib
import subprocess
from pathlib import Path

REPO_NAME  = 'VisDrone2019-Multi-Object-Detection-and-Tracking'
WORK_DIR   = f'/kaggle/working/{REPO_NAME}'
BASE_INPUT = '/kaggle/input/datasets/krantiprakash/dataset/Dataset'
BASE_WORK  = '/kaggle/working/Dataset'

if not os.path.exists(WORK_DIR):
    raise RuntimeError(f'Repo not found at {WORK_DIR}. Ensure Cell 2 ran successfully.')

os.chdir(WORK_DIR)

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import configs.paths
importlib.reload(configs.paths)
from configs.paths import get_paths, verify_paths

# copy images and annotations for DET-train, DET-val, MOT-val
dirs_to_copy = [
    ('VisDrone2019-DET-train', ['images', 'annotations']),
    ('VisDrone2019-DET-val',   ['images', 'annotations']),
    ('VisDrone2019-MOT-val',   ['sequences', 'annotations']),
]

for split, subdirs in dirs_to_copy:
    for subdir in subdirs:
        src = f'{BASE_INPUT}/{split}/{subdir}'
        dst = f'{BASE_WORK}/{split}/{subdir}'
        Path(f'{BASE_WORK}/{split}').mkdir(parents=True, exist_ok=True)
        if not os.path.exists(dst):
            print(f'Copying {split}/{subdir} ...')
            result = subprocess.run(
                ['cp', '-r', src, dst],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                raise RuntimeError(f'Copy failed for {src}: {result.stderr}')
            print(f'Done    : {dst}')
        else:
            print(f'Exists  : {dst}')

print('\nVerifying dataset paths after copy...')
paths = get_paths()
verify_paths(paths)

print('\nRunning convert_to_yolo.py...')
!python src/data/convert_to_yolo.py

In [ ]:
# Cell 6: Fine-tune YOLO26x on VisDrone-DET
import os

REPO_NAME = 'VisDrone2019-Multi-Object-Detection-and-Tracking'
WORK_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(WORK_DIR):
    raise RuntimeError(f'Repo not found at {WORK_DIR}. Ensure Cell 2 ran successfully.')

os.chdir(WORK_DIR)

!python src/detection/train_yolo.py

In [ ]:
# Cell 7: Verify outputs available for download
import os
import sys
import importlib

REPO_NAME = 'VisDrone2019-Multi-Object-Detection-and-Tracking'
WORK_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(WORK_DIR):
    raise RuntimeError(f'Repo not found at {WORK_DIR}. Ensure Cell 2 ran successfully.')

os.chdir(WORK_DIR)

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import configs.paths
importlib.reload(configs.paths)
from configs.paths import get_paths

paths        = get_paths()
download_dir = paths['out_detection'] / 'download'

print('Files available for download:')
if download_dir.exists():
    for f in sorted(download_dir.iterdir()):
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name:<40} {size_mb:.1f} MB')
else:
    print(f'  Download folder not found: {download_dir}')

In [ ]:
# Cell 8: Run detection evaluation on DET-test
import os
from pathlib import Path

REPO_NAME  = 'VisDrone2019-Multi-Object-Detection-and-Tracking'
WORK_DIR   = f'/kaggle/working/{REPO_NAME}'
MODEL_PATH = '/kaggle/input/datasets/krantiprakash/visdrone-weight/weights/best.pt'
TEST_IMAGES= '/kaggle/input/datasets/krantiprakash/visdrone2019-det-test/VisDrone2019-DET-test/images'
TEST_ANN   = '/kaggle/input/datasets/krantiprakash/visdrone2019-det-test/VisDrone2019-DET-test/annotations'

if not os.path.exists(WORK_DIR):
    raise RuntimeError(f'Repo not found at {WORK_DIR}. Ensure Cell 2 ran successfully.')

for name, path in {
    'best.pt'          : MODEL_PATH,
    'test images'      : TEST_IMAGES,
    'test annotations' : TEST_ANN,
}.items():
    if os.path.exists(path):
        size = f'{os.path.getsize(path)/1e6:.1f} MB' if path.endswith('.pt') else f'{len(os.listdir(path))} files'
        print(f'  found   : {name} ({size})')
    else:
        raise RuntimeError(f'Missing: {name} -> {path}')

os.chdir(WORK_DIR)
!python src/detection/evaluate_det.py